# Lab 03 - Cloud Evaluation (solution)

Reference notebook: `4 - cloud evaluation/4.1 - Cloud Evaluation.ipynb`.

Local evaluation is perfect for a prototype on small data. When the dataset grows, when results must be
tracked in a project, or when evaluation becomes a CI/CD gate, the run moves to the cloud: Foundry hosts
the compute, stores the dataset as a versioned asset and keeps every run observable in the portal.

## Step 0 - Configuration

In [ ]:
import os, sys, json, time, warnings
from pprint import pprint

from lab_utils import load_settings

warnings.filterwarnings("ignore")

TO_BE_EVALUATED_FILE = "./assets/synthetic_dataset_cloud.jsonl"
FILE_VERSION = "1.0"

settings = load_settings(verbose=True)
credential = settings["credential"]
foundry_project_endpoint = settings["foundry_project_endpoint"]
judge_deployment = settings["azure_evaluation_compatible_deployment_name"]

if not foundry_project_endpoint:
    raise ValueError("FOUNDRY_PROJECT_ENDPOINT is required for cloud evaluation")

In [ ]:
from azure.ai.projects import AIProjectClient

project_client = AIProjectClient(endpoint=foundry_project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print("Project client ready")

## Step 1 - Inspect the dataset

`assets/synthetic_dataset_cloud.jsonl` holds 10 records with `query`, `context`, `response` and
`ground_truth`. Those four fields are the contract between the dataset and the testing criteria.

In [ ]:
with open(TO_BE_EVALUATED_FILE, encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

print(f"{len(records)} records, fields: {list(records[0].keys())}\n")
pprint(records[0])

## Step 2 - Helper functions

Dataset versions are immutable: uploading `1.0` twice fails. `upload_dataset` validates the JSONL,
then walks forward until it finds a version that is genuinely free and visible in the project listing.

In [ ]:
from azure.core.exceptions import ResourceExistsError, ResourceNotFoundError


def increase_version(version: str) -> str:
    """Return the next version by incrementing its numeric final part."""
    parts = version.split(".")
    if len(parts) > 1:
        return f"{'.'.join(parts[:-1])}.{int(parts[-1]) + 1}"
    return str(int(version) + 1)


def validate_jsonl(file_path: str) -> None:
    """Fail early on malformed datasets: the cloud run would fail much later."""
    with open(file_path, encoding="utf-8") as jsonl_file:
        for line_number, line in enumerate(jsonl_file, start=1):
            if not line.strip():
                raise ValueError(f"Blank line found in {file_path} at line {line_number}")
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON in {file_path} at line {line_number}, column {error.colno}: {error.msg}"
                ) from error
            if not isinstance(record, dict):
                raise ValueError(f"Expected a JSON object in {file_path} at line {line_number}")


def upload_dataset(project_client, file_path: str, initial_version: str,
                   max_version_attempts: int = 20, listing_checks: int = 12,
                   listing_check_interval: int = 5):
    """Upload a dataset using the first usable version that appears in the project listing."""
    if file_path.lower().endswith(".jsonl"):
        validate_jsonl(file_path)

    file_name = file_path.rsplit("/", 1)[-1].rsplit(".", 1)[0]
    file_version = initial_version

    for _ in range(max_version_attempts):
        version_is_listed = any(
            d.name == file_name and d.version == file_version for d in project_client.datasets.list()
        )
        try:
            project_client.datasets.get(name=file_name, version=file_version)
            version_exists = True
        except ResourceNotFoundError:
            version_exists = False

        if version_is_listed:
            print(f"Dataset {file_name} version {file_version} is already listed; trying the next version")
            file_version = increase_version(file_version)
            continue

        if version_exists:
            project_client.datasets.delete(name=file_name, version=file_version)
            print(f"Deletion requested for unlisted dataset {file_name} version {file_version}")
            file_version = increase_version(file_version)
            continue

        try:
            dataset = project_client.datasets.upload_file(
                name=file_name, file_path=file_path, version=file_version
            )
        except ResourceExistsError:
            print(f"Dataset {file_name} version {file_version} appeared during upload; trying the next version")
            file_version = increase_version(file_version)
            continue

        for _ in range(listing_checks):
            if any(d.name == dataset.name and d.version == dataset.version
                   for d in project_client.datasets.list()):
                return dataset
            time.sleep(listing_check_interval)

        raise RuntimeError(
            f"Dataset {dataset.name} version {dataset.version} was uploaded but did not appear in the "
            f"project listing after {listing_checks * listing_check_interval} seconds."
        )

    raise RuntimeError(f"No usable version found for {file_name} after {max_version_attempts} attempts")

## Step 3 - Upload the dataset as a project asset

In [ ]:
file_dataset = upload_dataset(
    project_client=project_client,
    file_path=TO_BE_EVALUATED_FILE,
    initial_version=FILE_VERSION,
)

data_id = file_dataset.id

print(f"""The file {file_dataset.name} version {file_dataset.version}
has been uploaded by {file_dataset["systemData"]["createdBy"]}
at {file_dataset["dataUri"]}""")

## Step 4 - Describe the data and choose the evaluators

`DataSourceConfigCustom` declares the schema of one item; the testing criteria then map each evaluator
input to a dataset field with the `{{item.<field>}}` syntax.

Note the difference between the criteria:

* `builtin.f1_score` is **deterministic** - no model, it compares `response` and `ground_truth`;
* `builtin.groundedness` and `builtin.relevance` are **AI judges** - they take an
  `initialization_parameters.model`;
* `builtin.violence` is a **safety** evaluator backed by the Foundry service.

In [ ]:
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileID,
)
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator

data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query": {"type": "string"},
            "response": {"type": "string"},
            "context": {"type": "string"},
            "ground_truth": {"type": "string"},
        },
        "required": ["query", "response", "context", "ground_truth"],
    },
    include_sample_schema=True,
)

In [ ]:
testing_criteria = [
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="f1_score",
        evaluator_name="builtin.f1_score",
        data_mapping={
            "response": "{{item.response}}",
            "ground_truth": "{{item.ground_truth}}",
        },
    ),
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="groundedness",
        evaluator_name="builtin.groundedness",
        initialization_parameters={"model": judge_deployment},
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{item.response}}",
        },
    ),
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="relevance",
        evaluator_name="builtin.relevance",
        initialization_parameters={"model": judge_deployment},
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{item.response}}",
        },
    ),
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="violence",
        evaluator_name="builtin.violence",
        initialization_parameters={"model": judge_deployment},
        data_mapping={
            "query": "{{item.query}}",
            "response": "{{item.response}}",
        },
    ),
]

len(testing_criteria)

## Step 5 - Create the evaluation and start the run

The **evaluation object** defines *what* is measured and is reusable; the **run** applies it to one
dataset. Several runs under the same evaluation are directly comparable - this is what turns evaluation
into a regression test.

In [ ]:
eval_object = openai_client.evals.create(
    name="Workshop cloud evaluation - lab 03",
    data_source_config=data_source_config,
    testing_criteria=testing_criteria,
)

print(f"Evaluation created: {eval_object.id}")

In [ ]:
eval_run = openai_client.evals.runs.create(
    eval_id=eval_object.id,
    name="Cloud evaluation run",
    data_source=CreateEvalJSONLRunDataSourceParam(
        type="jsonl",
        source=SourceFileID(type="file_id", id=data_id),
    ),
)

print(f"Evaluation run created: {eval_run.id}")
print(f"Initial status: {eval_run.status}")

## Step 6 - Poll the run and read the results

In [ ]:
while True:
    eval_run = openai_client.evals.runs.retrieve(run_id=eval_run.id, eval_id=eval_object.id)
    if eval_run.status in ("completed", "failed", "canceled"):
        break
    print(f"Evaluation status: {eval_run.status}")
    time.sleep(5)

print(f"Final status: {eval_run.status}")
print(f"Report URL:   {eval_run.report_url}")

In [ ]:
if eval_run.status == "completed":
    output_items = list(
        openai_client.evals.runs.output_items.list(run_id=eval_run.id, eval_id=eval_object.id)
    )
    print(f"{len(output_items)} output items\n")

    for item in output_items[:3]:
        payload = item.model_dump(exclude_none=True, warnings=False)
        scores = {r.get("name"): r.get("score") for r in payload.get("results", [])}
        print(f'- passed={payload.get("status")} | {scores}')

### Checkpoint

You have a versioned dataset in the project, a reusable evaluation definition, a completed run and a
report URL to open in the portal. The optional part below plugs in **your own** evaluators.

## Step 7 (optional) - Use the custom evaluators published in Lab 01

If you completed the optional publishing step of Lab 01, `friendliness_evaluator` and
`response_length_score_evaluator` are in the project catalog. They are referenced by **name and
version**, and their `initialization_parameters` must match the schema declared at publish time.

Check the version numbers in the Foundry portal (or in the output of Lab 01) before running this.

In [ ]:
custom_criteria = testing_criteria + [
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="friendliness",
        evaluator_name="friendliness_evaluator",
        evaluator_version="1",
        initialization_parameters={"deployment_name": judge_deployment, "threshold": 3},
        data_mapping={"response": "{{item.response}}"},
    ),
    TestingCriterionAzureAIEvaluator(
        type="azure_ai_evaluator",
        name="response_length",
        evaluator_name="response_length_score_evaluator",
        evaluator_version="1",
        initialization_parameters={"deployment_name": judge_deployment, "pass_threshold": 0.5},
        data_mapping={"answer": "{{item.response}}"},
    ),
]

mixed_eval = openai_client.evals.create(
    name="Mixed cloud evaluation - builtin + custom",
    data_source_config=data_source_config,
    testing_criteria=custom_criteria,
)

mixed_run = openai_client.evals.runs.create(
    eval_id=mixed_eval.id,
    name="Mixed cloud evaluation run",
    data_source=CreateEvalJSONLRunDataSourceParam(
        type="jsonl", source=SourceFileID(type="file_id", id=data_id)
    ),
)

print(f"Mixed run: {mixed_run.id} ({mixed_run.status})")

## Step 8 (optional) - Evaluate a dataset you generated yourself

Take the JSONL produced in Lab 02 (`02-dataset-generation/generated_datasets/`), reshape it into the
`query` / `context` / `response` / `ground_truth` schema and run the same evaluation. This closes the
loop: **generate -> evaluate -> compare**.

In [ ]:
# Sketch: adapt the paths to the dataset you generated in Lab 02
#
# source = "../02-dataset-generation/generated_datasets/simulated_conversations.jsonl"
# records = [json.loads(line) for line in open(source, encoding="utf-8") if line.strip()]
#
# rows = []
# for record in records:
#     messages = record["messages"]
#     for i, message in enumerate(messages):
#         if message["role"] == "assistant" and i > 0 and messages[i - 1]["role"] == "user":
#             rows.append({
#                 "query": messages[i - 1]["content"],
#                 "context": messages[i - 1].get("context", ""),
#                 "response": message["content"],
#                 "ground_truth": "",
#             })
#
# with open("assets/generated_dataset_cloud.jsonl", "w", encoding="utf-8") as f:
#     for row in rows:
#         f.write(json.dumps(row, ensure_ascii=False) + "\n")
#
# then upload_dataset(...) and repeat steps 5-6

## Wrap-up

* Cloud evaluation = versioned dataset + reusable evaluation definition + comparable runs.
* Criteria are declarative: field mapping errors, not code, are the usual cause of failures.
* Deterministic, AI-judge and safety evaluators can be mixed in the same run.
* Custom evaluators published to the catalog behave exactly like the built-in ones.